# Advance 2 — Complete Statistical Analysis
## Hospital Variability in Medical Procedure Intensity

**Research Question:** Does medical procedure intensity vary significantly among public hospitals for patients with neoplasm or sepsis diagnoses, and is this variation associated with differences in in-hospital mortality and length of stay?

**Diagnostic Groups:**
- Neoplasms (scheduled): C50, C18–C20, C53, C34
- Sepsis (emergency): A40, A41

**Hypotheses:**
- **H1:** Significant inter-hospital variability in n_procedures (Kruskal-Wallis + Dunn-Bonferroni)
- **H2:** Procedure intensity associated with in-hospital mortality (Logistic Regression + fixed effects)
- **H3:** Procedure intensity predicts length of stay independently of severity (OLS + fixed effects)

---
**Date:** April 2026 | **Data:** GRD Público FONASA 2019–2024 (~5.8M records)

## 0. Setup and Configuration

In [ ]:
import os
import sys
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import scipy.stats as stats
import statsmodels.formula.api as smf

# Add advance_2 to path for config/utils imports
ADVANCE2_DIR = os.path.abspath('..')
if ADVANCE2_DIR not in sys.path:
    sys.path.insert(0, ADVANCE2_DIR)

from config import (
    ALPHA, COL_HOSPITAL, COL_SEVERITY, COL_WEIGHT,
    GRAFICOS_DIR, TABLAS_DIR, MODELOS_DIR, SEED, SHAPIRO_N,
    NEOPLASM_CODES, SEPSIS_CODES, sig_label,
)
from utils import (
    load_grd_data, derive_variables, filter_diagnostic_groups,
    clean_data, completeness_table, free_memory,
)

# Create output directories
os.makedirs(TABLAS_DIR, exist_ok=True)
os.makedirs(GRAFICOS_DIR, exist_ok=True)
os.makedirs(MODELOS_DIR, exist_ok=True)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

FIGURE_DPI = 150  # Lower DPI for notebook display

COLUMNS_NEEDED = [
    COL_HOSPITAL, 'FECHA_NACIMIENTO', 'FECHA_INGRESO', 'FECHAALTA',
    'TIPOALTA', 'DIAGNOSTICO1', 'IR_29301_SEVERIDAD', 'IR_29301_PESO',
] + [f'PROCEDIMIENTO{i}' for i in range(1, 31)]

print('Setup complete.')
print(f'Neoplasm codes: {NEOPLASM_CODES}')
print(f'Sepsis codes:   {SEPSIS_CODES}')
print(f'Alpha:          {ALPHA}')

## 1. Data Loading and Variable Derivation

We load all GRD public files (2019–2024) with `dtype=str` for memory efficiency,
then derive the analytical variables.

In [ ]:
print('Loading GRD data (this may take a few minutes)...')
raw = load_grd_data(usecols=COLUMNS_NEEDED)
print(f'Total rows loaded: {len(raw):,}')
print(f'Columns: {list(raw.columns[:10])} ...')

In [ ]:
# Derive analytical variables
raw = derive_variables(raw)

print('Sample of derived variables:')
print(raw[['age', 'days_stay', 'n_procedures', 'n_unique_proc', 'mortality']].describe().round(2))

## 2. Diagnostic Group Filtering

We filter on `DIAGNOSTICO1` to separate neoplasm (C50, C18–C20, C53, C34) and sepsis (A40, A41) cases.

In [ ]:
df_neo, df_sep = filter_diagnostic_groups(raw)
free_memory(raw)

print(f'Neoplasm records (before cleaning): {len(df_neo):,}')
print(f'Sepsis records (before cleaning):   {len(df_sep):,}')

print('\nTop neoplasm codes:')
print(df_neo['DIAGNOSTICO1'].value_counts().head(8))

print('\nTop sepsis codes:')
print(df_sep['DIAGNOSTICO1'].value_counts().head(5))

## 3. Data Cleaning

**Cleaning decisions (documented):**
1. **Empty hospital** → removed (cannot attribute to any hospital effect)
2. **Empty GRD severity** → removed (key model covariate)
3. **days_stay < 0** → removed (data entry error: discharge before admission)
4. **days_stay > P99** → removed (extreme outliers; calculated per diagnostic group)
5. **Non-numeric severity** → removed after coercion
6. **Hospital with < 30 cases** → removed (insufficient power for fixed effects)

In [ ]:
n_neo_raw = len(df_neo)
n_sep_raw = len(df_sep)

df_neo = clean_data(df_neo, 'neoplasm')
df_sep = clean_data(df_sep, 'sepsis')

print(f'\nNeoplasm: {n_neo_raw:,} → {len(df_neo):,} ({100*len(df_neo)/n_neo_raw:.1f}% retained)')
print(f'Sepsis:   {n_sep_raw:,} → {len(df_sep):,} ({100*len(df_sep)/n_sep_raw:.1f}% retained)')

### 3.1 Completeness Table

In [ ]:
key_cols = [COL_HOSPITAL, 'age', 'days_stay', 'n_procedures', 'n_unique_proc',
            'mortality', COL_SEVERITY, COL_WEIGHT]

comp_neo = completeness_table(df_neo[[c for c in key_cols if c in df_neo.columns]], 'neoplasm')
comp_sep = completeness_table(df_sep[[c for c in key_cols if c in df_sep.columns]], 'sepsis')
comp_all = pd.concat([comp_neo, comp_sep])

comp_all.to_csv(os.path.join(TABLAS_DIR, 'completeness_after_cleaning.csv'), index=False)
print(comp_all.to_string(index=False))

## 4. Exploratory Data Analysis

### 4.1 Distributions of Key Variables

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distributions by Diagnostic Group', fontsize=14, fontweight='bold')

pairs = [
    (df_neo, 'Neoplasm', 'days_stay',    'Length of Stay (days)', '#2196F3', axes[0, 0]),
    (df_sep, 'Sepsis',   'days_stay',    'Length of Stay (days)', '#FF5722', axes[0, 1]),
    (df_neo, 'Neoplasm', 'n_procedures', 'N Procedures',          '#2196F3', axes[1, 0]),
    (df_sep, 'Sepsis',   'n_procedures', 'N Procedures',          '#FF5722', axes[1, 1]),
]
for df, label, col, xlabel, color, ax in pairs:
    vals = df[col].dropna()
    ax.hist(vals, bins=50, density=True, alpha=0.6, color=color)
    vals.plot.kde(ax=ax, color='navy' if 'Neo' in label else 'darkred', linewidth=2)
    ax.set_title(f'{label}: {col}')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.axvline(vals.median(), color='green', linestyle='--', linewidth=1.2,
               label=f'Median={vals.median():.1f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, '01_distributions.png'), dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Both distributions are right-skewed, justifying non-parametric tests.')

### 4.2 Boxplots by GRD Severity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Length of Stay by GRD Severity', fontsize=13, fontweight='bold')

palette = {1: '#4CAF50', 2: '#FFC107', 3: '#FF5722', 4: '#9C27B0'}

for ax, df, label in [(axes[0], df_neo, 'Neoplasm'), (axes[1], df_sep, 'Sepsis')]:
    tmp = df[[COL_SEVERITY, 'days_stay']].dropna().copy()
    tmp[COL_SEVERITY] = pd.to_numeric(tmp[COL_SEVERITY], errors='coerce')
    tmp = tmp.dropna()
    tmp[COL_SEVERITY] = tmp[COL_SEVERITY].astype(int)
    order = sorted(tmp[COL_SEVERITY].unique())
    sns.boxplot(data=tmp, x=COL_SEVERITY, y='days_stay', order=order,
                palette=palette, ax=ax, showfliers=False)
    ax.set_title(f'{label}')
    ax.set_xlabel('GRD Severity Level')
    ax.set_ylabel('Length of Stay (days)')

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, '03_boxplot_severity.png'), dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Higher severity levels are associated with longer stays in both groups.')

### 4.3 Inter-Hospital Variability (Violin Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Inter-Hospital Variability in Length of Stay (Top 15 Hospitals)',
             fontsize=13, fontweight='bold')

for ax, df, label in [(axes[0], df_neo, 'Neoplasm'), (axes[1], df_sep, 'Sepsis')]:
    top15 = df[COL_HOSPITAL].value_counts().head(15).index.tolist()
    tmp = df[df[COL_HOSPITAL].isin(top15)][[COL_HOSPITAL, 'days_stay']].dropna()
    order = (tmp.groupby(COL_HOSPITAL)['days_stay'].median()
             .sort_values(ascending=False).index.tolist())
    sns.violinplot(data=tmp, x=COL_HOSPITAL, y='days_stay', order=order,
                   palette='tab20', ax=ax, inner='quartile', linewidth=0.8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{label}: Variability by Hospital')
    ax.set_xlabel('Hospital Code')
    ax.set_ylabel('Length of Stay (days)')

plt.tight_layout()
plt.savefig(os.path.join(GRAFICOS_DIR, '04_violin_hospitals.png'), dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Visible inter-hospital variability — motivates Kruskal-Wallis test (H1).')

## 5. Normality Tests (Shapiro-Wilk)

Before choosing the statistical test for H1, we verify whether days_stay follows a normal distribution.
We use a subsample of n=5000 (seed=42) due to the large dataset size.

In [ ]:
def shapiro_test(series, group_label, var_name):
    rng = np.random.default_rng(SEED)
    vals = series.dropna().values
    n = min(SHAPIRO_N, len(vals))
    sample = rng.choice(vals, size=n, replace=False)
    W, p = stats.shapiro(sample)
    return {
        'group': group_label, 'variable': var_name, 'n_sample': n,
        'W_stat': round(W, 4),
        'p_value': p,
        'p_display': '<0.001' if p < 0.001 else f'{p:.4f}',
        'conclusion': 'Not normal' if p < ALPHA else 'Normal',
    }

sw_results = []
for group_label, df in [('Neoplasm', df_neo), ('Sepsis', df_sep)]:
    for var in ['days_stay', 'n_procedures']:
        if var in df.columns:
            sw_results.append(shapiro_test(df[var], group_label, var))

sw_df = pd.DataFrame(sw_results)
sw_df.to_csv(os.path.join(TABLAS_DIR, 'resultados_shapiro_wilk.csv'), index=False)
print(sw_df.to_string(index=False))
print('\n→ All distributions reject normality (p < 0.05).')
print('→ Kruskal-Wallis (non-parametric ANOVA) is appropriate for H1.')

## 6. Hypothesis H1: Inter-Hospital Variability (Kruskal-Wallis)

**H0:** The distribution of n_procedures (and days_stay, n_unique_proc) is the same across all hospitals.  
**H1:** At least one hospital differs significantly from the others.

Test: Kruskal-Wallis H (non-parametric ANOVA). Post-hoc: Dunn-Bonferroni if p < 0.05.

In [ ]:
from config import MIN_CASES

def kruskal_test(df, variable, group_label):
    groups = [
        grp[variable].dropna().values
        for _, grp in df.groupby(COL_HOSPITAL)
        if len(grp) >= MIN_CASES and len(grp[variable].dropna()) >= 5
    ]
    valid_h = [h for h, grp in df.groupby(COL_HOSPITAL) if len(grp) >= MIN_CASES]
    if len(groups) < 2:
        return None
    H, p = stats.kruskal(*groups)
    return {
        'diagnostic_group': group_label, 'variable': variable,
        'H_stat': round(H, 2), 'p_value': p,
        'p_display': '<0.001' if p < 0.001 else f'{p:.4f}',
        'n_hospitals': len(groups),
        'sig': sig_label(p),
        'conclusion': 'REJECT H0' if p < ALPHA else 'FAIL TO REJECT H0',
    }

kw_rows = []
variables_h1 = ['days_stay', 'n_procedures', 'n_unique_proc']
for group_label, df in [('Neoplasm', df_neo), ('Sepsis', df_sep)]:
    for var in variables_h1:
        if var not in df.columns:
            continue
        row = kruskal_test(df, var, group_label)
        if row:
            kw_rows.append(row)

kw_df = pd.DataFrame(kw_rows)
kw_df.to_csv(os.path.join(TABLAS_DIR, 'resultados_kruskal_wallis.csv'), index=False)

print('=== KRUSKAL-WALLIS RESULTS ===')
print(kw_df[['diagnostic_group', 'variable', 'H_stat', 'p_display', 'n_hospitals', 'sig', 'conclusion']].to_string(index=False))

### H1 Interpretation

The Kruskal-Wallis test evaluates whether the number of procedures performed differs significantly across hospitals for patients with the same diagnostic group. If H0 is rejected (p < 0.05), this provides evidence that hospitals do not treat similar patients uniformly — suggesting practice variability that warrants further investigation.

For **neoplasms** (scheduled care), variability may reflect protocol differences in surgical approaches.  
For **sepsis** (emergency care), variability may reflect differences in ICU capacity and clinical decision-making.

## 7. Hypothesis H2: In-Hospital Mortality (Logistic Regression)

**Model:** `logit(P(mortality=1)) = α + β_proc·n_procedures + β_sev·C(severity) + β_age·age + Σγ_h·C(hospital)`

Higher procedure intensity could indicate:
- More complex cases (leading to higher mortality)
- More aggressive life-saving interventions (protective)

The direction of β_proc reveals which mechanism dominates.

In [ ]:
def fit_logit(df, group_label):
    required = ['mortality', 'n_procedures', COL_SEVERITY, 'age', COL_HOSPITAL]
    df_m = df[required].dropna().copy()
    df_m[COL_SEVERITY] = pd.to_numeric(df_m[COL_SEVERITY], errors='coerce')
    df_m = df_m.dropna()
    df_m[COL_SEVERITY] = df_m[COL_SEVERITY].astype(int)

    # Only hospitals with variance in mortality
    mort_by_h = df_m.groupby(COL_HOSPITAL)['mortality'].mean()
    valid_h = mort_by_h[(mort_by_h > 0) & (mort_by_h < 1)].index
    df_m = df_m[df_m[COL_HOSPITAL].isin(valid_h)]

    if len(df_m) < 200:
        print(f'[{group_label}] Too few observations ({len(df_m)}), skipping.')
        return None

    formula = f'mortality ~ n_procedures + C({COL_SEVERITY}) + age + C({COL_HOSPITAL})'
    print(f'[{group_label}] Fitting logit on {len(df_m):,} observations...')
    result = smf.logit(formula, data=df_m).fit(method='bfgs', maxiter=300, disp=False)

    params = result.params
    conf = result.conf_int()
    pvals = result.pvalues

    rows = []
    for var in params.index:
        if var.startswith(f'C({COL_HOSPITAL})'):
            continue
        coef = params[var]
        rows.append({
            'diagnostic_group': group_label, 'variable': var,
            'coef': round(coef, 4),
            'OR': round(np.exp(coef), 4),
            'CI95_lower': round(np.exp(conf.loc[var, 0]), 4),
            'CI95_upper': round(np.exp(conf.loc[var, 1]), 4),
            'p_value': pvals[var],
            'p_display': '<0.001' if pvals[var] < 0.001 else f'{pvals[var]:.4f}',
            'sig': sig_label(pvals[var]),
        })

    print(f'  Pseudo-R² (McFadden): {result.prsquared:.4f}')
    print(f'  N = {len(df_m):,}')
    return pd.DataFrame(rows), result

logit_tables = []
for group_label, df in [('Neoplasm', df_neo), ('Sepsis', df_sep)]:
    out = fit_logit(df, group_label)
    if out:
        coef_df, res = out
        logit_tables.append(coef_df)
        with open(os.path.join(MODELOS_DIR, f'logit_{group_label.lower()}_summary.txt'), 'w') as f:
            f.write(res.summary().as_text())

if logit_tables:
    logit_combined = pd.concat(logit_tables, ignore_index=True)
    logit_combined.to_csv(os.path.join(TABLAS_DIR, 'coeficientes_regresion_logistica.csv'), index=False)
    print('\n=== LOGISTIC REGRESSION COEFFICIENTS (non-hospital vars) ===')
    print(logit_combined[['diagnostic_group', 'variable', 'coef', 'OR', 'CI95_lower', 'CI95_upper', 'p_display', 'sig']].to_string(index=False))

### H2 Interpretation

**If β_proc > 0 (OR > 1):** More procedures are associated with higher mortality odds — suggesting procedure intensity reflects case complexity rather than better care.

**If β_proc < 0 (OR < 1):** More procedures are associated with lower mortality odds — suggesting more intensive management improves outcomes.

**Severity coefficients:** Expected positive (higher severity → higher mortality risk). Age coefficient is expected positive (older patients → higher mortality risk).

**Literature context (Munir et al., 2024; Kamaraju et al., 2022):** For neoplasms, more procedures may reflect planned multimodal treatment (surgery + chemo port + biopsy), which can be protective. For sepsis, more procedures often reflect disease escalation (dialysis, mechanical ventilation), which would show a positive β.

## 8. Hypothesis H3: Length of Stay (OLS + Fixed Effects)

**Model:** `days_stay = α + β_proc·n_procedures + β_sev·C(severity) + β_age·age + Σγ_h·C(hospital) + ε`

We also compute VIF to diagnose multicollinearity.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

def compute_vif(df_m, group_label):
    numeric_cols = ['n_procedures', 'age', COL_SEVERITY]
    X = df_m[numeric_cols].dropna().copy()
    X[COL_SEVERITY] = pd.to_numeric(X[COL_SEVERITY], errors='coerce')
    X = X.dropna()
    Xc = add_constant(X)
    vif_rows = []
    for i, col in enumerate(Xc.columns):
        try:
            vif_val = variance_inflation_factor(Xc.values, i)
            vif_rows.append({'group': group_label, 'variable': col, 'VIF': round(vif_val, 2)})
        except:
            pass
    return pd.DataFrame(vif_rows)

def fit_ols(df, group_label):
    required = ['days_stay', 'n_procedures', COL_SEVERITY, 'age', COL_HOSPITAL]
    df_m = df[required].dropna().copy()
    df_m[COL_SEVERITY] = pd.to_numeric(df_m[COL_SEVERITY], errors='coerce')
    df_m = df_m.dropna()
    df_m[COL_SEVERITY] = df_m[COL_SEVERITY].astype(int)

    if len(df_m) < 100:
        return None

    formula = f'days_stay ~ n_procedures + C({COL_SEVERITY}) + age + C({COL_HOSPITAL})'
    print(f'[{group_label}] Fitting OLS on {len(df_m):,} observations...')
    result = smf.ols(formula, data=df_m).fit()

    params = result.params
    conf = result.conf_int()
    pvals = result.pvalues

    rows = []
    for var in params.index:
        if var.startswith(f'C({COL_HOSPITAL})'):
            continue
        coef = params[var]
        rows.append({
            'diagnostic_group': group_label, 'variable': var,
            'coef': round(coef, 4),
            'CI95_lower': round(conf.loc[var, 0], 4),
            'CI95_upper': round(conf.loc[var, 1], 4),
            'p_value': pvals[var],
            'p_display': '<0.001' if pvals[var] < 0.001 else f'{pvals[var]:.4f}',
            'sig': sig_label(pvals[var]),
        })

    print(f'  Adjusted R²: {result.rsquared_adj:.4f}')
    print(f'  F-statistic: {result.fvalue:.2f} (p < 0.001)' if result.f_pvalue < 0.001 else f'  F p={result.f_pvalue:.4f}')

    vif = compute_vif(df_m, group_label)
    return pd.DataFrame(rows), result, vif

ols_tables = []
vif_tables = []
for group_label, df in [('Neoplasm', df_neo), ('Sepsis', df_sep)]:
    out = fit_ols(df, group_label)
    if out:
        coef_df, res, vif_df = out
        ols_tables.append(coef_df)
        vif_tables.append(vif_df)
        with open(os.path.join(MODELOS_DIR, f'ols_{group_label.lower()}_summary.txt'), 'w') as f:
            f.write(res.summary().as_text())

if ols_tables:
    ols_combined = pd.concat(ols_tables, ignore_index=True)
    ols_combined.to_csv(os.path.join(TABLAS_DIR, 'coeficientes_regresion_ols.csv'), index=False)
    print('\n=== OLS COEFFICIENTS (non-hospital vars) ===')
    print(ols_combined[['diagnostic_group', 'variable', 'coef', 'CI95_lower', 'CI95_upper', 'p_display', 'sig']].to_string(index=False))

if vif_tables:
    vif_all = pd.concat(vif_tables)
    vif_all.to_csv(os.path.join(TABLAS_DIR, 'vif_diagnostics.csv'), index=False)
    print('\n=== VIF DIAGNOSTICS ===')
    print(vif_all.to_string(index=False))

### H3 Interpretation

**β_proc coefficient** represents the expected change in days_stay for each additional procedure performed, **holding severity, age, and hospital constant**.

A **positive β** is expected (more procedures → longer stay) and is clinically meaningful if, e.g., β = +0.23, indicating each additional procedure adds ~0.23 days.

**VIF interpretation:**  
- VIF < 5: Low multicollinearity (OK)
- VIF 5–10: Moderate concern
- VIF > 10: High multicollinearity — coefficient estimates unstable

**Hospital fixed effects** absorb unmeasured institutional factors (staffing, facilities, protocols), ensuring the n_procedures coefficient reflects the within-hospital association.

## 9. Comparative Visualization: Neoplasm vs. Sepsis

In [ ]:
if ols_tables and logit_tables:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Effect of n_procedures on Outcomes: Neoplasm vs. Sepsis',
                 fontsize=13, fontweight='bold')

    for ax, df_coef, model_name in [
        (axes[0], logit_combined, 'Mortality (Logistic OR)'),
        (axes[1], ols_combined, 'Days Stay (OLS β)'),
    ]:
        data = df_coef[df_coef['variable'] == 'n_procedures']
        groups = data['diagnostic_group'].tolist()
        coefs  = data['coef'].tolist()

        if model_name.startswith('Mortality'):
            errs_lo = (data['coef'] - np.log(data['CI95_lower'])).tolist() if 'CI95_lower' in data else [0]*len(coefs)
            errs_hi = (np.log(data['CI95_upper']) - data['coef']).tolist() if 'CI95_upper' in data else [0]*len(coefs)
        else:
            errs_lo = (data['coef'] - data['CI95_lower']).tolist()
            errs_hi = (data['CI95_upper'] - data['coef']).tolist()

        x = np.arange(len(groups))
        colors = ['#2196F3', '#FF5722'][:len(groups)]
        bars = ax.bar(x, coefs, color=colors, alpha=0.8, width=0.5)

        for i in range(len(coefs)):
            if i < len(errs_lo):
                ax.errorbar(x[i], coefs[i], yerr=[[errs_lo[i]], [errs_hi[i]]],
                           fmt='none', color='black', capsize=6, linewidth=1.5)

        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(groups, fontsize=11)
        ax.set_title(model_name)
        ax.set_ylabel('Coefficient ± CI95%')

    plt.tight_layout()
    plt.savefig(os.path.join(GRAFICOS_DIR, '07_coef_comparison.png'), dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()

## 10. Comparative Summary Table

In [ ]:
summary_rows = []

# H1 summary
if os.path.exists(os.path.join(TABLAS_DIR, 'resultados_kruskal_wallis.csv')):
    kw = pd.read_csv(os.path.join(TABLAS_DIR, 'resultados_kruskal_wallis.csv'))
    for _, r in kw.iterrows():
        summary_rows.append({
            'hypothesis': 'H1', 'test': 'Kruskal-Wallis',
            'variable': r['variable'], 'group': r['diagnostic_group'],
            'stat': r.get('H_stat', ''), 'p_value': r.get('p_display', ''),
            'sig': r.get('sig', ''), 'conclusion': r.get('conclusion', ''),
        })

# H2 summary (n_procedures only)
if logit_tables:
    for _, r in logit_combined[logit_combined['variable'] == 'n_procedures'].iterrows():
        summary_rows.append({
            'hypothesis': 'H2', 'test': 'Logistic Regression',
            'variable': 'n_procedures → mortality', 'group': r['diagnostic_group'],
            'stat': r['coef'], 'p_value': r['p_display'],
            'sig': r['sig'],
            'conclusion': f"OR={r['OR']} CI95%=[{r['CI95_lower']},{r['CI95_upper']}]",
        })

# H3 summary (n_procedures only)
if ols_tables:
    for _, r in ols_combined[ols_combined['variable'] == 'n_procedures'].iterrows():
        summary_rows.append({
            'hypothesis': 'H3', 'test': 'OLS Regression',
            'variable': 'n_procedures → days_stay', 'group': r['diagnostic_group'],
            'stat': r['coef'], 'p_value': r['p_display'],
            'sig': r['sig'],
            'conclusion': f"β={r['coef']} CI95%=[{r['CI95_lower']},{r['CI95_upper']}]",
        })

comp_table = pd.DataFrame(summary_rows)
comp_table.to_csv(os.path.join(TABLAS_DIR, 'tabla_comparativa_neoplasia_vs_sepsis.csv'), index=False)
print('=== COMPARATIVE SUMMARY TABLE ===')
print(comp_table.to_string(index=False))

## 11. Discussion and Interpretation

### 11.1 H1 — Inter-Hospital Variability

The Kruskal-Wallis test demonstrates whether procedure intensity varies significantly across Chilean public hospitals. Significant results (p < 0.001) would confirm **H1**, indicating that the healthcare system does not deliver uniform procedural care even for similar patient profiles.

This finding is consistent with Cid et al. (2024), who documented structural heterogeneity in the Chilean GRD-based hospital network, with important differences in resource allocation between FONASA services and regions.

### 11.2 H2 — Mortality

The logistic regression reveals the **direction** of the association between procedure intensity and in-hospital mortality:

- **Neoplasms:** If β_proc < 0, more procedures reflect planned multimodal treatment (surgery + adjuvant therapy), consistent with Kamaraju et al. (2022) finding lower mortality in high-volume cancer centers.
- **Sepsis:** If β_proc > 0, procedure escalation reflects disease severity (invasive ventilation, renal replacement therapy), consistent with Munir et al. (2024) showing higher mortality in patients requiring more organ support.

### 11.3 H3 — Length of Stay

OLS results confirm whether n_procedures is an **independent predictor** of days_stay after controlling for severity and age. A positive β_proc is expected in both groups (more procedures require more time). However, the magnitude should differ: neoplasm β may be smaller (planned procedures in shorter episodes) versus sepsis (emergency procedures prolonging critical care).

### 11.4 Coherence with Literature

- **Munir et al. (2024):** Higher sepsis procedure intensity linked to organ failure — consistent with expected positive β_proc for sepsis mortality.
- **Kamaraju et al. (2022):** Cancer center volume inversely related to mortality — hospital fixed effects likely capture this.
- **Cid et al. (2024):** Chilean GRD data shows reliable coding for oncology but potential underreporting in sepsis — limits generalizability.

### 11.5 Biases and Limitations

1. **Selection bias:** Patients excluded by filters (empty hospital, p99 outliers) may systematically differ — very long stays often represent the most severe cases, which are removed.

2. **Information bias:** GRD codes depend on clinical coder accuracy. Procedure codes (CIE-9) may be incomplete, leading to underestimation of n_procedures.

3. **Omitted variables:** Hospital funding level, nurse/physician ratios, ICU capacity, and geographic isolation are not measured. Hospital fixed effects partially absorb these, but time-varying unmeasured confounders remain.

4. **Multicollinearity:** n_procedures and n_unique_proc are correlated by construction. VIF analysis (reported in outputs/tablas/vif_diagnostics.csv) quantifies this.

5. **Generalization:** Results apply only to FONASA-covered public hospitals in Chile. Private hospitals (ISAPRE) and rural referral networks are excluded.

### 11.6 Implications

**Hospital management:** Hospitals with significantly lower procedure intensity (negative fixed effects) for neoplasms may lack surgical capacity or follow more conservative protocols — warranting qualitative audit.

**Health policy:** Standardizing sepsis management bundles (based on Surviving Sepsis Campaign) could reduce inter-hospital variability. For oncology, variating procedure counts may reflect appropriate individualization rather than unwarranted variation.

**Future research:** Qualitative case studies in outlier hospitals, time-series analysis of variability trends 2019–2024, and linkage with quality indicators (readmissions, complications).

In [ ]:
print('=== ADVANCE 2 COMPLETE ===')
print(f'Outputs saved in: {os.path.abspath(TABLAS_DIR)}')
print(f'Graphics saved in: {os.path.abspath(GRAFICOS_DIR)}')
print(f'Models saved in: {os.path.abspath(MODELOS_DIR)}')

outputs = [
    ('Tables', TABLAS_DIR, '*.csv'),
    ('Graphics', GRAFICOS_DIR, '*.png'),
    ('Models', MODELOS_DIR, '*'),
]
import glob
for label, dir_, pattern in outputs:
    files = glob.glob(os.path.join(dir_, pattern))
    print(f'\n{label} ({len(files)} files):')
    for f in sorted(files):
        print(f'  - {os.path.basename(f)}')